# Vision Extraction Tests

Test Qwen2.5-VL-7B on a sample PISA PDF page.

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
from src.ingestion.vlm_extractor import VLMExtractor
from src.ingestion.pdf_processor import PDFProcessor
import asyncio, json

config_path = Path('../configs/model_configs.yaml')
prompt_path = Path('../configs/prompts/vision_extractor.txt')

extractor = VLMExtractor(config_path, prompt_path)
processor = PDFProcessor(dpi=200, max_pages_per_shard=3)

# Test on first PDF
pdfs = list(Path('../data/raw_pdfs').glob('*.pdf'))
print(f'Found {len(pdfs)} PDFs')

if pdfs:
    async def test():
        async for shard in processor.shard_pdf(pdfs[0]):
            print(f'Shard {shard.shard_index}: pages {shard.page_range_str}')
            records = extractor.extract_shard(shard.images, pdfs[0].name, shard.page_range_str)
            print(json.dumps(records, indent=2, ensure_ascii=False))
            break  # test first shard only
    asyncio.run(test())

## Check extraction quality

Expected fields: topic, question_summary (verbatim), correct_concept, common_misconceptions, key_vocabulary

In [ ]:
# Validate all records from JSONL
import jsonlines
records = list(jsonlines.open('../data/processed_jsonl/knowledge_base.jsonl'))
print(f'Total records: {len(records)}')
missing_fields = [r for r in records if not r.get('correct_concept')]
print(f'Records missing correct_concept: {len(missing_fields)}')
print('\nSample record:')
if records: print(json.dumps(records[0], indent=2, ensure_ascii=False))